Imports and environment

In [1]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
from IPython.display import Markdown, display
import json
from datetime import datetime
import re
from urllib.parse import urlsplit
from bs4 import BeautifulSoup
import os
import time

load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

print(api_key[:10])

fc-f48a711


Archive selection logic and slug extraction

In [ ]:
response = requests.get("https://www.morningbrew.com/api/sanity/issues/archive?end=3000")
print(response.status_code)
print(response.text[:500])

# Doing a simple request still doesn't work

In [ ]:
# Testing scraping with Firecrawl again

client = Firecrawl(api_key=api_key)

result = client.scrape(
    "https://www.morningbrew.com/api/sanity/issues/archive?end=3000",
    formats=["markdown"]
)

display(Markdown(result.markdown))

In [ ]:
display(result.markdown)

In [ ]:
# The relevant info is in a json code block so the characters need to be removed

result.markdown.strip("'`json\n")

In [ ]:
full_archive = json.loads(result.markdown.strip("'`json\n"))

In [8]:
# Testing datetime functionality

date = datetime.strptime((full_archive[0]["date"].split("T"))[0],"%Y-%m-%d").date()
date

datetime.date(2026, 8, 16)

In [6]:
cutoff = datetime.fromisoformat("2020-10-31").date()
cutoff

datetime.date(2020, 10, 31)

In [2]:
# Combining above cells to create function and store archive

def scrape_archive_metadata():
    client = Firecrawl(api_key=api_key)

    result = client.scrape(
        "https://www.morningbrew.com/api/sanity/issues/archive?end=3000",
        formats=["markdown"]
    )

    return json.loads(result.markdown.strip("'`json\n"))

full_archive_metadata = scrape_archive_metadata()


In [ ]:
# Define date cutoffs and valid weekdays

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday

# Function to process the date item into a usable Python date

def processed_date(item):
    return datetime.strptime((item["date"].split("T"))[0],"%Y-%m-%d").date()

# Extraction function with logic

def extract_slug(item):
    date = processed_date(item)
    if date.weekday() not in valid_weekdays:
        return None
    elif date.weekday() == 3 and date >= first_thursday:
        if date >= first_worded_slug:
            return item["slug"]
        else:
            return item["slug"].replace("/", "%2F")
    elif date.weekday() == 5 and date < first_thursday:
        return item["slug"].replace("/", "%2F")


In [ ]:
# Testing the extraction function on an individual edition

print(extract_slug(full_archive_metadata[64]))

'its-just-field'

In [ ]:
# Looping over all editions and printing to check it is working as intended

slugs = []

for item in full_archive_metadata:
    if extract_slug(item):
        print(extract_slug(item))
        slugs.append(extract_slug(item))
        #print(processed_date(item).weekday())
    else:
        continue

print(len(slugs))

Extracting raw URLs from Morning Brew links

In [ ]:
response = requests.get("https://links.morningbrew.com/c/U6n?mblid=f92dc3fd9d90&mbnlid=slV5a3vuYndT7PDVh24Aj8")
print(response.text)

In [ ]:
# I tried Beautiful Soup first
soup = BeautifulSoup(response.text, 'html.parser')
soup.find('link', attrs={'rel': 'canonical'}).get('href')

In [ ]:
# But then realised that his works too (I've now changed it to .headers.get("Location") instead of .url as some websites hung without response)

response = requests.get("https://links.morningbrew.com/c/U6q?mblid=0d6a557bf158&mbnlid=slV5a3vuYndT7PDVh24Aj8")
print(response.url)
o = urlsplit(response.url)
o._replace(query="").geturl()

In [ ]:
# Testing an unresponsive URL and how to get the right destination

response = requests.get("https://links.morningbrew.com/c/Ox0?mblid=19b17c4e99da&mbnlid=J4PVUnjbNhxkECeFjuBu2S", allow_redirects=False, timeout=10)
print(response.status_code)
print(response.headers.get("Location"))
url = response.headers.get("Location")
urlsplit(url)._replace(query="").geturl()

302
https://www.cbc.ca/news/canada/toronto/iceman-release-date-drake-ice-pile-9.7171479?utm_campaign=mb&utm_medium=newsletter&utm_source=morning_brew


'https://www.cbc.ca/news/canada/toronto/iceman-release-date-drake-ice-pile-9.7171479'

Archiving

In [95]:
iso_date = date.isoformat()
print(f"iso date: {iso_date}")

# for edition in archive:
#     if edition["date"] == iso_date:
#         exists = True
#         exit
#     else:
#         exists = False

exists = any(edition["date"] == iso_date for edition in archive)
exists


iso date: 2025-01-16


True

# The full process

In [ ]:
# Constants and initial steps (just testing on one edition)

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday
index = 86
edition_metadata = full_archive_metadata[index]

def processed_date(edition_metadata):
    return datetime.strptime((edition_metadata["date"].split("T"))[0],"%Y-%m-%d").date()

id = edition_metadata["_id"]
date = processed_date(edition_metadata)
edition_metadata

{'_id': '0rMtBzRcQOSiUlzbv9S3Ff',
 'date': '2026-05-21T06:00:00-04:00',
 'previewText': "Elon Musk's space company made its IPO plans public...",
 'slug': 'spacex-ipo-ready-for-blastoff',
 'subjectLine': 'SpaceX IPO ready for blastoff',
 'title': None,
 'websitePreviewText': None}

In [ ]:
data_path = "../data"

with open(f"{data_path}/archive.json", "r") as f:
    archive = json.load(f)

archive

In [106]:
archive = []
quizzes = []

In [ ]:
# Check to see if edition exits, scrape it if not and store in archive

data_path = "../data"

def edition_condition_and_store(date, index, full_archive_metadata):

    # Check to see if archive exists, if not create blank list
    if os.path.isfile(f"{data_path}/archive.json"):
        with open(f"{data_path}/archive.json", "r") as f:
            archive = json.load(f)
    else:
        archive = []
    if os.path.isfile(f"{data_path}/quizzes.json"):
            with open(f"{data_path}/quizzes.json", "r") as f:
                quizzes = json.load(f)
    else:
        quizzes = []

    iso_date = date.isoformat()

    if any(archive_edition["date"] == iso_date for archive_edition in archive):
        return f"Edition for {iso_date} found"
    else:
        process_edition(index, full_archive_metadata, date, archive, quizzes)

    with open(f"{data_path}/archive.json", "w") as f:
        json.dump(archive, f, indent=2)

    with open(f"{data_path}/quizzes.json", "w") as f:
        json.dump(quizzes, f, indent=2)
    
edition_condition_and_store(date, index, full_archive_metadata)

In [ ]:
quizzes

In [ ]:
# Refactoring the single edition processing into a function so I can add logic to determine whether it already exists

def process_edition(index, full_archive_metadata, date, archive, quizzes):
    edition_metadata = full_archive_metadata[index]
    id = edition_metadata["_id"]
    slug = extract_slug(edition_metadata, date)
    scraped_edition = scrape_edition(slug)
    print("Edition scraped")
    edition = {
        "_id" : id,
        "date" : date.isoformat(),
        "slug" : slug,
        "content" : scraped_edition.markdown
    }
    questions_and_answers = filter_parts(scraped_edition)
    questions_and_answers = clean_q4_and_answer(questions_and_answers)

    quiz = {
            "_id": id,
            "edition_date": date.isoformat(),
            "source_url": f"https://www.morningbrew.com/issues/{slug}",
            "headlines": [],
            "answer": questions_and_answers[4]
        }

    print("Extracting headline text and URLs")
    for index, item in enumerate(questions_and_answers):
        if item[0].isalpha():
            continue
        else:
            headline, url = url_headline_extract(index, questions_and_answers)
            quiz["headlines"].append({"text": headline, "url": url})

    print(json.dumps(quiz, indent=2))

    return archive.append(edition), quizzes.append(quiz)

    # time.sleep(6)

# process_edition(index, full_archive_metadata, date)


In [12]:
iso_date = "2025-08-28"
archive = [
  {
    "_id": "4VloMqduImvtGYAjN2PfX9",
    "date": "2025-08-21",
    "slug": "on-target",
    "content": "[Skip to main content](https://www.morningbrew.com/issues/on-target#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### \u2615On Target\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| The struggling retailer is getting a new CEO... |\n\n[![](https://media.sailthru.com/5z8/1k3/3/k/5c9296dc6e8c2.png)](http://www.morningbrew.com/)\n\n| |     |     |\n| --- | --- |\n| |     |     |\n| --- | --- |\n| August 21, 2025 | [View Online](https://links.morningbrew.com/c/Dyq?mbnlid=4VloMqduImvtGYAjN2PfX9) \\| [Sign Up](https://links.morningbrew.com/c/BJ_?mbnlid=4VloMqduImvtGYAjN2PfX9) \\| [Shop](https://links.morningbrew.com/c/r-?mbnlid=4VloMqduImvtGYAjN2PfX9) |\n\n|     |\n| --- |\n| [![Morning Brew](https://cdn.sanity.io/images/bl383u0v/production/65e2f7af234829dba0f11606a9a1430efab65892-1350x330.png?w=900&fit=max)](https://links.morningbrew.com/c/BK0?mbnlid=4VloMqduImvtGYAjN2PfX9) |\n\n|     |     |\n| --- | --- |\n| Presented By | [![Pacaso](https://cdn.sanity.io/images/bl383u0v/production/e684a2c32903549ae850c79271a27c312acf1f13-1040x240.png?w=600&q=100&auto=format)](https://links.morningbrew.com/c/DxC?lp=logo&mbadid=b4831bbb560ac722855d1a156eb6618e&mbadv=a&mbnlid=4VloMqduImvtGYAjN2PfX9) |\n\n|     |\n| --- |\n| ![](https://em-content.zobj.net/thumbs/120/apple/237/studio-microphone_1f399.png)**Good morning.** Mad Dog Matty Merritt, Dave \u201cThe Loze\u201d Lozo, and Molly Molly Oxen Free Liebergall are coming to you LIVE from the top of the newsletter (airhorn blast, airhorn blast, airhorn blast). We\u2019ve got a very special newsletter for our readers today, but first...Markets (echo sound effect), Markets (echo sound effect), Markets\u2026<br>_\u2014Matty Merritt, Dave Lozo, Molly Liebergall, Abby Rubenstein_ | |\n\n| |     |\n| --- |\n| ### MARKETS |\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n|  |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/e4a85d043f857cc8adaca9d546c099f6d7db37ba-44x39.png) | Nasdaq | 21,172.86 | |     |\n| --- |\n| -0.67% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/e4a85d043f857cc8adaca9d546c099f6d7db37ba-44x39.png) | S&P | 6,395.78 | |     |\n| --- |\n| -0.24% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/6097ec1c26b17401a61bfbfe4440c75ebfea1886-44x39.png) | Dow | 44,938.31 | |     |\n| --- |\n| +0.04% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/e4a85d043f857cc8adaca9d546c099f6d7db37ba-44x39.png) | 10-Year | 4.296% | |     |\n| --- |\n| -1.0 bps | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/6097ec1c26b17401a61bfbfe4440c75ebfea1886-44x39.png) | Bitcoin | $114,395.11 | |     |\n| --- |\n| +0.83% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/e4a85d043f857cc8adaca9d546c099f6d7db37ba-44x39.png) | Palantir | $156.01 | |     |\n| --- |\n| -1.10% | | |\n| |     |     |\n| --- | --- |\n| Data is provided by | [![](https://cdn.sanity.io/images/bl383u0v/production/61a0dac0b53d6766459afdca784ae8fe604f17ed-720x102.png?h=32&q=100&auto=format)](https://links.morningbrew.com/c/k2s?mbnlid=4VloMqduImvtGYAjN2PfX9) |\n\n\\*Stock data as of market close, cryptocurrency data as of 6:00pm ET. [Here's what these numbers mean.](https://links.morningbrew.com/c/BK2?mbnlid=4VloMqduImvtGYAjN2PfX9) |\n| - **Markets:**Tech stocks [tanked](https://links.morningbrew.com/c/Dyr?mblid=4ff941208d5a&mbnlid=4VloMqduImvtGYAjN2PfX9) yesterday, dragging down the market as investors continued to worry about an AI bubble\u2014in part because AI poster child Sam Altman told reporters last week that there was one, and MIT recently published a report saying 95% of corporate AI pilots were failing to create a return. There are also concerns about interest rates as the financial world waits for JPow\u2019s Jackson Hole speech tomorrow.<br>- **Stock spotlight:** Amid the broader AI sell-off, Palantir had its sixth losing day in a row. It\u2019s the [worst streak](https://links.morningbrew.com/c/Dys?mblid=c046957a395d&mbnlid=4VloMqduImvtGYAjN2PfX9) for the secretive software company since April 2024. |\n| **Markets Sponsored by Pacaso**<br>**Early-stage opportunity ends Sept. 18:** $270m+. That\u2019s how much Pacaso raised from the VCs behind Uber, Airbnb, and more, as well as 10k+ regular people. [You can invest too, but only until Sept. 18](https://links.morningbrew.com/c/DxD?lp=text1&mbadid=8df191c2a2f5748c8b370951babfecc7&mbadv=a&mblid=da9bfc374213&mbnlid=4VloMqduImvtGYAjN2PfX9). | |\n| --- |\n\n|     |\n| --- |\n| ### RETAIL\n\n|     |\n| --- |\n| # [Target gets a new CEO, Wall Street not impressed](https://links.morningbrew.com/c/Dyt?mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n|\n|\n\n[![Target store with shoppers leaving](https://cdn.sanity.io/images/bl383u0v/production/7a086b8a39c6ddccd84f52b59d11f4e5dd8441be-1500x1000.jpg?w=668&q=70&auto=format)](https://links.morningbrew.com/c/Dyt?mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n_Scott Olson/Getty Images_\n\n|     |\n| --- |\n| An inspirational tale for the interns currently wrapping up their summer of filing and happy hours: Target [announced](https://links.morningbrew.com/c/Dyu?mblid=3e877b53bdfb&mbnlid=4VloMqduImvtGYAjN2PfX9) yesterday that COO Michael Fiddelke, who started at the company with an internship two decades ago, will be stepping into the CEO role. The shakeup at the top comes after years of declining sales for the chain due to steep competition and customer boycotts.<br>Fiddelke will take the helm in February, so current CEO Brian Cornell will have [one more](https://links.morningbrew.com/c/Dyv?mblid=4e73502fa38d&mbnlid=4VloMqduImvtGYAjN2PfX9) office holiday party in the role before they release him from his corner office ~~to wander around the tumbler cup aisle until retirement~~ (he\u2019ll become executive chair of Target\u2019s board of directors).<br>The soon-to-be chief exec comes to the job as the ultimate Target insider. Fiddelke started at Target as a finance intern in 2003, scoring a permanent analyst role the following year. Since then, he\u2019s worked in the company\u2019s merchandising, operations, and HR departments before stepping into the COO and CFO roles, the latter of which he held from 2019 to 2024. Now, his plans include reestablishing Target as an elevated shopping experience by focusing on a \u201cstyle and design North Star.\u201d<br>**It\u2019s not the turnaround Wall Street was hoping for\u2026** Target\u2019s stock ended trading down more than 6% after the announcement. Analysts were hoping an outsider would step into the role to help shake the retailer out of its current funk.<br>## He\u2019s got a big job ahead<br>Target spent the pandemic doing gangbusters as consumers snapped up tons of tech and housewares, but when inflation started to rise, all those discretionary item sales dried up. In the first quarter of 2022, Target\u2019s profits had dropped 52% from a year earlier. Since then:<br>- Target has been losing ground to rivals like Walmart and Amazon over the last few years.<br>- The chain also faced [backlash](https://links.morningbrew.com/c/Dyw?mblid=382188883a03&mbnlid=4VloMqduImvtGYAjN2PfX9) from right-wing groups over its Pride merch.<br>- Additionally, its profits and store visits have been impacted by [consumer boycotts](https://links.morningbrew.com/c/Dyx?mblid=0894093bf6fa&mbnlid=4VloMqduImvtGYAjN2PfX9) after the company rolled back its DEI initiatives earlier this year.<br>**Now what?** Yesterday, Target reported declining sales as well as decreasing store and website traffic for fiscal Q2\u2014though it still managed to beat Wall Street\u2019s expectations. Incoming CEO Fiddelke tried to rally the troops after the news broke by calling on employees to go the extra mile to help Target get back on top. _\u2014MM_ |\n| |     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/cab2ac077d288f7ba1df3571cb0ec28ce8f32b86-357x357.png?w=50&fit=max)](https://links.morningbrew.com/c/Dyy?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/7372d7c942cc95144c43e3004730ca0cf8a23025-96x96.png)](https://links.morningbrew.com/c/Dyz?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/774271f5d2278014f1af0759475660e1761384eb-357x357.png?w=50&fit=max)](https://links.morningbrew.com/c/DyA?mbnlid=4VloMqduImvtGYAjN2PfX9) | | | |\n\n|     |\n| --- |\n| ### Presented By Pacaso\n\n# [**Major investors bet on this \u201cunlisted\u201d stock**](https://links.morningbrew.com/c/DxC?lp=title&mbadid=b4831bbb560ac722855d1a156eb6618e&mbadv=a&mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n[![Pacaso](https://cdn.sanity.io/images/bl383u0v/production/6662d65a78476fc06a4e3a70568ccb9580d5f769-1200x800.png?w=668&q=90&auto=format)](https://links.morningbrew.com/c/DxC?lp=image&mbadid=b4831bbb560ac722855d1a156eb6618e&mbadv=a&mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n|     |\n| --- |\n| When the former Zillow exec who sold his first company for $120m starts a new venture, people notice. That\u2019s why [the same VC firms that backed Uber, Venmo, and eBay also invested in Pacaso](https://links.morningbrew.com/c/DxC?lp=text1&mbadid=b4831bbb560ac722855d1a156eb6618e&mbadv=a&mblid=a466160ec24d&mbnlid=4VloMqduImvtGYAjN2PfX9).<br>Disrupting the real estate industry once again, Pacaso\u2019s streamlined platform offers co-ownership of premier properties, revamping the $1.3t vacation-home market.<br>And it works. By handing keys to 2k+ happy homeowners, [Pacaso has already made $110m+ in gross profits](https://links.morningbrew.com/c/DxC?lp=text2&mbadid=b4831bbb560ac722855d1a156eb6618e&mbadv=a&mblid=5dd37fa8da2d&mbnlid=4VloMqduImvtGYAjN2PfX9) in their operating history.<br>Now, after 41% YoY gross profit growth last year alone, they recently reserved the Nasdaq ticker PCSO.<br>And you can join them as an early-stage investor for just $2.90/share. Just don\u2019t wait. [Pacaso\u2019s investment opportunity officially ends Sept. 18](https://links.morningbrew.com/c/DxC?lp=text3&mbadid=b4831bbb560ac722855d1a156eb6618e&mbadv=a&mblid=036bc17f1755&mbnlid=4VloMqduImvtGYAjN2PfX9). | |\n\n|     |\n| --- |\n| ### WORLD\n\n|     |\n| --- |\n| # [Tour de headlines](https://www.morningbrew.com/issues/on-target\\#)\n\n|\n|\n\n![Federal Reserve Governor Lisa Cook ](https://cdn.sanity.io/images/bl383u0v/production/4facafb919a4f8a92d9069e1ca2a5bf83d6e036b-1500x1000.jpg?w=668&q=70&auto=format)\n\n_Drew Angerer/Getty Images_\n\n|     |\n| --- |\n| **![](https://em-content.zobj.net/thumbs/120/apple/237/bank_1f3e6.png) Trump calls for Fed Governor Lisa Cook to resign.** President Trump posted on Truth Social yesterday that Lisa Cook, a Biden-appointed Federal Reserve governor, \u201cmust resign, now!!!\u201d after the leader of the Federal Housing Finance Agency, Bill Pulte, claimed she submitted fraudulent information to banks on mortgage loan documents. Citing anonymous sources, the Wall Street Journal [reported](https://links.morningbrew.com/c/DyB?mblid=2e7f56afd625&mbnlid=4VloMqduImvtGYAjN2PfX9) that Trump is considering firing Cook for cause. It\u2019s his latest swipe at the independence of the central bank, which hasn\u2019t lowered interest rates like the president wants. Pulte, a Trump ally who has made similar allegations against prominent critics of the president, asked the DOJ to investigate Cook, alleging she had requested mortgages in Atlanta and Michigan and claimed both properties as her primary residence. Cook said she would not be \u201cbullied to step down.\u201d<br>**![](https://em-content.zobj.net/thumbs/120/apple/237/page-with-curl_1f4c3.png) \u201cAlmost all\u201d Fed officials supported leaving rates unchanged, per meeting minutes.** The minutes from the Fed\u2019s July meeting [are out](https://links.morningbrew.com/c/DyC?mblid=69ec33d73b80&mbnlid=4VloMqduImvtGYAjN2PfX9), and they suggest that the two officials who voted against keeping interest rates steady were alone in that effort\u2014but it still represented the first time since 1993 that more than one Fed governor dissented from a rate decision. The minutes show that most Fed officials remained more concerned about the possibility of rising inflation than problems in the labor market, though that could change going forward since the meeting took place two days before disappointing [jobs data](https://links.morningbrew.com/c/CIl?mblid=35ed1a15a7fc&mbnlid=4VloMqduImvtGYAjN2PfX9) came out. The issue is politically fraught since President Trump has been pushing for a cut (see above).<br>**![](https://em-content.zobj.net/thumbs/120/apple/237/ballot-box-with-ballot_1f5f3.png) Texas House passes redistricting bill.** After a fight that involved Democratic lawmakers fleeing the state and being forced to have police escorts when they returned, the Texas House yesterday [passed](https://links.morningbrew.com/c/Dz6?mblid=93c5bbc25082&mbnlid=4VloMqduImvtGYAjN2PfX9) the redistricting bill President Trump requested, sending it to the state\u2019s Senate for a final vote. If the Senate passes the bill unchanged, it will advance to the governor\u2019s desk, but if it alters the bill, the two versions will have to be reconciled. The new map, which could add five seats for Republicans in Congress, could set off a flurry of gerrymandering efforts around the country. California\u2019s Governor has vowed to respond with his own redistricting, a move that former President Barack Obama endorsed yesterday. _\u2014AR_ | | |\n\n|     |\n| --- |\n| ### ENTERTAINMENT\n\n|     |\n| --- |\n| # [ESPN\u2019s new streaming app is for cord-nevers](https://links.morningbrew.com/c/DyD?mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n|\n|\n\n[![ESPN logo on a phone screen](https://cdn.sanity.io/images/bl383u0v/production/852a9ad95f72153bc8839d8b0ef83bf60157f649-1500x1000.jpg?w=668&q=70&auto=format)](https://links.morningbrew.com/c/DyD?mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n_Thomas Fuller/SOPA Images/LightRocket/Getty Images_\n\n|     |\n| --- |\n| There\u2019s a new way to watch Pat McAfee interview guests sitting in their cars. ESPN\u2019s eponymously named direct-to-consumer app [launched](https://links.morningbrew.com/c/DyE?mblid=6d48531af22b&mbnlid=4VloMqduImvtGYAjN2PfX9) today, finally making all of the Worldwide Leader\u2019s traditional TV products available on streaming for people who don\u2019t pay to watch cable television.<br>- ESPN, the TV channel, is on the lowest basic cable tier, which means everyone paying for cable, including people who say \u201csportsball,\u201d already pays for ESPN.<br>- In 2011, ESPN was in 100 million homes and had generated hundreds of billions in revenue over three decades through a subscription/advertising model that was heavily insulated from cord-cutting. But as of today, ESPN is down to 61 million subscribers, per the NYT, as Americans increasingly flee from their cumbersome cable packages.<br>**Bundles:** The standalone ESPN app is $29.99 per month; it will stay the same price when bundled with Disney+ and Hulu for a year before it jumps to $35.99. Disney is betting that a bundle will [reduce churn](https://links.morningbrew.com/c/DyF?mblid=421cf476f5fb&mbnlid=4VloMqduImvtGYAjN2PfX9).<br>**Separations:** The NFL bought a 10% stake in ESPN earlier this month. Not long after, a documentary about Colin Kaepernick from Spike Lee four years in the making was [scrapped](https://links.morningbrew.com/c/DyG?mblid=20cf36afcce5&mbnlid=4VloMqduImvtGYAjN2PfX9), signaling that investigative journalism at the expense of the NFL may be hard to find on ESPN.<br>**Bottom line:** The new app, which also has interactive AI [features](https://links.morningbrew.com/c/DyH?mblid=3935cb383ab2&mbnlid=4VloMqduImvtGYAjN2PfX9), primarily targets cord-nevers, not necessarily cord-cutters. Anyone already paying for ESPN on cable can access the app for free. _\u2014DL_ |\n| |     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/cab2ac077d288f7ba1df3571cb0ec28ce8f32b86-357x357.png?w=50&fit=max)](https://links.morningbrew.com/c/DyI?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/7372d7c942cc95144c43e3004730ca0cf8a23025-96x96.png)](https://links.morningbrew.com/c/DyJ?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/774271f5d2278014f1af0759475660e1761384eb-357x357.png?w=50&fit=max)](https://links.morningbrew.com/c/DyK?mbnlid=4VloMqduImvtGYAjN2PfX9) | | | |\n\n|     |\n| --- |\n| ### Together With Fidelity Private Shares\u2120\n\n[![Fidelity Private Shares\u2120](https://cdn.sanity.io/images/bl383u0v/production/1776e85364e13e6f26954adb66e4d2abca338070-450x300.jpg?w=668&q=90&auto=format)](https://links.morningbrew.com/c/DxH?lp=image&mbadid=122caa64193eb678fa8cfc54348deb3c&mbadv=a&mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n|     |\n| --- |\n| **Venture to thrive.** Founders are a dime a dozen these days, so to really wow investors, you should over-prepare. That\u2019s where [Fidelity Private Shares](https://links.morningbrew.com/c/DxH?lp=text1&mbadid=122caa64193eb678fa8cfc54348deb3c&mbadv=a&mblid=734c137062f9&mbnlid=4VloMqduImvtGYAjN2PfX9)\u2019 cap table management tool comes in. Show up and show out with a living document that can prove you\u2019re ready to play to win. | |\n\n|     |\n| --- |\n| ### CRYPTO\n\n|     |\n| --- |\n| # [Wyoming launches its own digital currency](https://links.morningbrew.com/c/DyL?mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n|\n|\n\n[![GIF of spinning coin with buffalo stamp](https://cdn.sanity.io/images/bl383u0v/production/d29759a258907144016716cb157d7a60184514eb-700x467.gif?w=668&q=70&auto=format)](https://links.morningbrew.com/c/DyL?mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n_Anna Kim_\n\n|     |\n| --- |\n| The state with the smallest population and only two escalators just opened up a new frontier in crypto. Wyoming [released](https://links.morningbrew.com/c/DyM?mblid=67be0c642adf&mbnlid=4VloMqduImvtGYAjN2PfX9) the first state-backed stablecoin in the US this week, fittingly named Frontier Stable Token, or FRNT.<br>One year after Wyoming [announced](https://links.morningbrew.com/c/DyN?mblid=685772e3c111&mbnlid=4VloMqduImvtGYAjN2PfX9) that it wouldn\u2019t wait around for a federally issued stablecoin, the state\u2019s new currency debuted on Ethereum, Solana, and several other blockchains.<br>- As stablecoins frequently are, it\u2019s backed by tangible assets. In this case, that\u2019s cash and short-term Treasuries to keep the coin pegged to the US dollar.<br>- But in a public service twist, interest generated by the Wyoming reserves that back FRNT will go toward public school funding, officials have pledged.<br>FRNT may also avoid federal oversight. The recently signed [GENIUS Act](https://links.morningbrew.com/c/BJK?mblid=ce258b1b4c13&mbnlid=4VloMqduImvtGYAjN2PfX9), which introduced a regulatory framework for stablecoins, appears to apply to businesses but not state governments, according to Bloomberg.<br>**State officials envision use cases like\u2026** brokering public-private contracts (e.g., an energy deal) in FRNT, or locals using the coin to pay for their morning coffee.<br>**The Cowboy State is having a busy week:** The SEC\u2019s new crypto-friendly chair, as well as Eric Trump and other people of crypto influence, attended the Wyoming Blockchain Symposium over the past few days. Top economists and policymakers from around the world are also [arriving](https://links.morningbrew.com/c/DyO?mblid=1c47c6448293&mbnlid=4VloMqduImvtGYAjN2PfX9) for the 48th annual Jackson Hole Economic Symposium, where Fed Chair Jerome Powell is expected to hint at his interest-rate intentions in a speech tomorrow. _\u2014ML_ |\n| |     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/cab2ac077d288f7ba1df3571cb0ec28ce8f32b86-357x357.png?w=50&fit=max)](https://links.morningbrew.com/c/DyP?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/7372d7c942cc95144c43e3004730ca0cf8a23025-96x96.png)](https://links.morningbrew.com/c/DyQ?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/774271f5d2278014f1af0759475660e1761384eb-357x357.png?w=50&fit=max)](https://links.morningbrew.com/c/DyR?mbnlid=4VloMqduImvtGYAjN2PfX9) | | | |\n\n|     |\n| --- |\n| ### STAT\n\n|     |\n| --- |\n| # [Prime number: New No. 1](https://www.morningbrew.com/issues/on-target\\#)\n\n|\n|\n\n![Images from the Focus Friend app](https://cdn.sanity.io/images/bl383u0v/production/780ce8d5e84d06207d176518e1c3dc47df1fdaa8-1500x1000.jpg?w=668&q=70&auto=format)\n\n_Illustration: Morning Brew Design, Photos: Screenshot of Focus Friend_\n\n|     |\n| --- |\n| A bean who judges you for doomscrolling just surpassed ChatGPT as the No. 1 [most downloaded](https://links.morningbrew.com/c/DyS?mblid=c3045b20bd07&mbnlid=4VloMqduImvtGYAjN2PfX9) free app on Apple\u2019s App Store. The app, Focus Friend, comes from Hank Green, a YouTuber with 2.47 million subscribers, and it uses his genius for grabbing people\u2019s attention to get us all to look away from a screen for once.<br>In Green\u2019s words, the app succeeds in \u201chacking your brain\u201d by adding a little bean to your phone who is trying to finish knitting projects. The bean gets distracted and looks sad if you use your apps, so the only way to help it out is to keep off your phone until the timer you set ends. Plus:<br>- There are in-app purchases available, but no ads. \u201cIt\u2019s about letting people be in control of their attention, not selling their attention to someone else,\u201d Green said in a video.<br>- It also doesn\u2019t collect or share user data.<br>It\u2019s not the first app based on the Pomodoro Technique of punctuating focused work time with short breaks, but it\u2019s probably the first to top the charts on the strength of a busy anthropomorphized legume. _\u2014AR_ | | |\n\n|     |\n| --- |\n| ### Together With Vuori\n\n[![Vuori](https://cdn.sanity.io/images/bl383u0v/production/3de2aed2fca4be47d21501da63871b328b93a501-1434x1522.png?w=668&q=90&auto=format)](https://links.morningbrew.com/c/DxE?lp=image&mbadid=9a4a0f0fedca6b8e09a08791a3a904db&mbadv=a&mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n|     |\n| --- |\n| **Buttery soft and brand-new.** Vuori\u2019s [AllTheFeels\u2122 leggings](https://links.morningbrew.com/c/DxF?lp=text1&mbadid=9a4a0f0fedca6b8e09a08791a3a904db&mbadv=a&mblid=3d7b6de770fb&mbnlid=4VloMqduImvtGYAjN2PfX9) feel light as a feather and are fit for any fitness routine. Their bestselling performance essentials have been revamped with new colors, new prints, and unlimited comfiness for your downward-facing dog. Check out the styles. PS: New customers [get 20% off their first order](https://links.morningbrew.com/c/DxE?lp=text2&mbadid=9a4a0f0fedca6b8e09a08791a3a904db&mbadv=a&mblid=debca7703aeb&mbnlid=4VloMqduImvtGYAjN2PfX9) at email sign-up. | |\n\n|     |\n| --- |\n| ### NEWS\n\n|     |\n| --- |\n| # [What else is brewing](https://www.morningbrew.com/issues/on-target\\#)\n\n|\n|\n\n|     |\n| --- |\n| - [The Israeli military](https://links.morningbrew.com/c/DyT?mblid=be29f4a72eef&mbnlid=4VloMqduImvtGYAjN2PfX9) is calling up about 60,000 reservists ahead of its planned ground offensive to take Gaza City, a plan that has sparked protests in Israel and international opposition.<br>- [Hurricane Erin](https://links.morningbrew.com/c/Dz7?mblid=caaecdcc2d47&mbnlid=4VloMqduImvtGYAjN2PfX9) is bringing \u201clife-threatening currents\u201d along the East Coast.<br>- [Google](https://links.morningbrew.com/c/DyU?mblid=c5a32f1cc3ce&mbnlid=4VloMqduImvtGYAjN2PfX9) rolled out its Pixel 10 device lineup with a big focus on AI in its phones.<br>- [A federal judge](https://links.morningbrew.com/c/DyV?mblid=d4d15dd33bad&mbnlid=4VloMqduImvtGYAjN2PfX9) has temporarily blocked a Texas law that would require public school classrooms to display the Ten Commandments.<br>- [Hertz](https://links.morningbrew.com/c/DyW?mblid=ca1ca4ead2c5&mbnlid=4VloMqduImvtGYAjN2PfX9) will sell its used rental cars via Amazon.<br>- [Transcripts](https://links.morningbrew.com/c/DyX?mblid=d6b97bf03c6b&mbnlid=4VloMqduImvtGYAjN2PfX9) from the sex trafficking case against Jeffrey Epstein will remain sealed after a federal judge in New York denied the Justice Department\u2019s request to release them.<br>- [Walmart](https://links.morningbrew.com/c/DyY?mblid=1b7de0494388&mbnlid=4VloMqduImvtGYAjN2PfX9) recalled frozen shrimp that may be radioactive. | | |\n\n|     |\n| --- |\n| ### RECS\n\n|     |\n| --- |\n| ![To-do list banner](https://cdn.sanity.io/images/bl383u0v/production/762cbc4c673b8613a0de1bf6711a4fc25276f7a4-1350x330.png?w=668&q=70&auto=format)\n\n|     |\n| --- |\n| ![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**Sip:** This is the only [water bottle](https://links.morningbrew.com/c/DyZ?mblid=6d94554d48b1&mbnlid=4VloMqduImvtGYAjN2PfX9) you need. It\u2019s the perfect size, easy to clean, and comes in so many colors.\\*\\*<br>![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**Look up:** Check out [this atlas](https://links.morningbrew.com/c/Dy-?mblid=10dfdef3bbe2&mbnlid=4VloMqduImvtGYAjN2PfX9) of solar eclipses.<br>![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**Watch:** Level up your [s\u2019mores](https://links.morningbrew.com/c/Dy_?mblid=a375621e58c9&mbnlid=4VloMqduImvtGYAjN2PfX9) game before campfire season ends.<br>![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**They\u2019re all good:** See the [most popular dog breed](https://links.morningbrew.com/c/Dz0?mblid=23ba253751ea&mbnlid=4VloMqduImvtGYAjN2PfX9) in your state.<br>**![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png) Nada, zip, zilch:** Credit card balances gotcha down? [Put interest to rest until 2027](https://links.morningbrew.com/c/BT2?mblid=3f347df801d0&mbnlid=4VloMqduImvtGYAjN2PfX9) with a 0% intro APR card. FinanceBuzz\u2019s credit card experts found top options to help you save money and [ditch high interest for good](https://links.morningbrew.com/c/BT2?mblid=275ecf7ebc3c&mbnlid=4VloMqduImvtGYAjN2PfX9).\\*<br>_\\*A message from our sponsor. \\*\\*This is a product recommendation from our writers. When you buy through this link, Morning Brew may earn a commission._ | | |\n\n|     |\n| --- |\n| ### PLAY\n\n|     |\n| --- |\n| [![Games available from Morning Brew](https://cdn.sanity.io/images/bl383u0v/production/3524949da8d220051120175f97246671d52e0b38-2700x660.png?w=668&q=70&auto=format)](https://links.morningbrew.com/c/AGP?mbnlid=4VloMqduImvtGYAjN2PfX9)\n\n|     |\n| --- |\n| **Brew Mini:** If you took the test that gives you 400 points for writing your name, you\u2019re more than ready to dominate [today\u2019s Mini](https://links.morningbrew.com/c/Dz1?mblid=7990ab42d9f0&mbnlid=4VloMqduImvtGYAjN2PfX9).<br>## Three Headlines and a Lie<br>Three of these headlines are real and one is faker than your plans to go to bed early before a 6am flight. Can you spot the odd one out?<br>1. Justin Bieber impersonator [tricks](https://links.morningbrew.com/c/Dz2?mblid=9cf1daafa385&mbnlid=4VloMqduImvtGYAjN2PfX9) Vegas club into letting him perform, runs up $10k bar tab<br>2. NASA\u2019s Webb telescope [finds](https://links.morningbrew.com/c/Dz3?mblid=db9615bd089f&mbnlid=4VloMqduImvtGYAjN2PfX9) a new tiny moon around Uranus<br>3. Playing R&B in enclosure led to record number of capuchin monkey [births](https://links.morningbrew.com/c/Dz4?mblid=ce4a1d98493b&mbnlid=4VloMqduImvtGYAjN2PfX9) at Brazil zoo<br>4. Your late-night cheese fix might be [linked](https://links.morningbrew.com/c/Dz5?mblid=26b209bc3fd7&mbnlid=4VloMqduImvtGYAjN2PfX9) to your nightmares | | |\n\n|     |\n| --- |\n| ### SHARE THE BREW\n\n|     |\n| --- |\n| |     |\n| --- |\n| [Share Morning Brew](https://links.morningbrew.com/c/sH?mbnlid=4VloMqduImvtGYAjN2PfX9) with your friends, acquire free Brew swag, and then acquire more friends as a result of your fresh Brew swag.<br>We\u2019re saying we\u2019ll give you free stuff and more friends if you share a link. One link.<br> [![](https://cdn.sanity.io/images/bl383u0v/production/e8c0d668fd2ce584105a14629f9409b29ddffc91-3840x2160.png?w=640&q=60)](https://links.morningbrew.com/c/sH?mbnlid=4VloMqduImvtGYAjN2PfX9) <br>**Your referral count:** 5<br> [Click to Share](https://links.morningbrew.com/c/sH?mbnlid=4VloMqduImvtGYAjN2PfX9) <br>Or copy & paste your referral link to others:<br>[morningbrew.com/daily/r/?kid=9ec4d467](https://links.morningbrew.com/c/7tc?mbnlid=4VloMqduImvtGYAjN2PfX9) | | |\n\n|     |\n| --- |\n| ### ANSWER\n\n|     |\n| --- |\n| |     |\n| --- |\n| We made up the one about the capuchin monkeys.<br>Word of the Day<br>Today\u2019s Word of the Day is: anthropomorphized, meaning \u201ca nonhuman thing with human characteristics ascribed to it.\u201d Thanks to Dianne Marek from Mukwonago, WI, for the uncanny suggestion. Submit another Word of the Day [here](https://links.morningbrew.com/c/aVn?mblid=019531dc83d7&mbnlid=4VloMqduImvtGYAjN2PfX9). | | |\n\n| |     |\n| --- |\n| **\u2722 A Note From Pacaso**<br>This is a paid advertisement for Pacaso\u2019s Regulation A offering. Please read the offering circular at [invest.pacaso.com](https://links.morningbrew.com/c/vTS?mblid=9645be5d6401&mbnlid=4VloMqduImvtGYAjN2PfX9). Reserving the ticker symbol is not a guarantee that the company will go public. Listing on the Nasdaq is subject to approvals. | |\n| --- |\n\n|     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/365b8a21162a71a71c4190f5b1b6e2acd45a212f-96x96.png)](https://links.morningbrew.com/c/sJ?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/bc850f15719e478323219cbfe24640074ab0c306-96x96.png)](https://links.morningbrew.com/c/sK?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/4be3fbb7e388adf000065f63e01e069db9fdd60a-96x96.png)](https://links.morningbrew.com/c/sL?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/6af774b2e94fc201bff9ad736da144c7f1f8468a-96x96.png)](https://links.morningbrew.com/c/sM?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/ccfbd021e2000f116305d0f75c4e95dbdb2996ba-96x96.png)](https://links.morningbrew.com/c/49z?mbnlid=4VloMqduImvtGYAjN2PfX9)[![](https://cdn.sanity.io/images/bl383u0v/production/56835eaa4b1e8e7314950c4862d5dfccc9fc410c-96x96.png)](https://links.morningbrew.com/c/sO?mbnlid=4VloMqduImvtGYAjN2PfX9) |\n\n|     |\n| --- |\n| Written by [Abigail Rubenstein](https://links.morningbrew.com/c/t2R?mbnlid=4VloMqduImvtGYAjN2PfX9), [Matty Merritt](https://links.morningbrew.com/c/sKN?mbnlid=4VloMqduImvtGYAjN2PfX9), and [Dave Lozo](https://links.morningbrew.com/c/sI-?mbnlid=4VloMqduImvtGYAjN2PfX9)<br>Was this email forwarded to you? Sign up [here](https://links.morningbrew.com/c/sJ8?mbnlid=4VloMqduImvtGYAjN2PfX9).<br>**Take The Brew to work**<br>- Marketers: [Marketing Brew](https://links.morningbrew.com/c/6Ok?mbnlid=4VloMqduImvtGYAjN2PfX9)<br>- Corporate: [CFO Brew](https://links.morningbrew.com/c/6Ol?mbnlid=4VloMqduImvtGYAjN2PfX9)[HR Brew](https://links.morningbrew.com/c/6Om?mbnlid=4VloMqduImvtGYAjN2PfX9)[Revenue Brew](https://links.morningbrew.com/c/yJ0?mbnlid=4VloMqduImvtGYAjN2PfX9)<br>- Tech: [IT Brew](https://links.morningbrew.com/c/6On?mbnlid=4VloMqduImvtGYAjN2PfX9)[Tech Brew](https://links.morningbrew.com/c/COA?mbnlid=4VloMqduImvtGYAjN2PfX9)<br>- Retailers: [Retail Brew](https://links.morningbrew.com/c/6Op?mbnlid=4VloMqduImvtGYAjN2PfX9)<br>- Healthcare: [Healthcare Brew](https://links.morningbrew.com/c/6Oq?mbnlid=4VloMqduImvtGYAjN2PfX9)<br>**Get smarter in just 5 minutes**<br>- Money & Career: [Money With Katie](https://links.morningbrew.com/c/s20?mbnlid=4VloMqduImvtGYAjN2PfX9)[Bossy](https://links.morningbrew.com/c/s21?mbnlid=4VloMqduImvtGYAjN2PfX9)[Brew Markets](https://links.morningbrew.com/c/s22?mbnlid=4VloMqduImvtGYAjN2PfX9)[The Playbook](https://links.morningbrew.com/c/s23?mbnlid=4VloMqduImvtGYAjN2PfX9)<br>**Interested in podcasts?**<br>- Check out ours [here](https://links.morningbrew.com/c/s24?mbnlid=4VloMqduImvtGYAjN2PfX9) |\n| [ADVERTISE](https://links.morningbrew.com/c/BK5?mbnlid=4VloMqduImvtGYAjN2PfX9) // [CAREERS](https://links.morningbrew.com/c/BK6?mbnlid=4VloMqduImvtGYAjN2PfX9) // [SHOP](https://links.morningbrew.com/c/t3?mbnlid=4VloMqduImvtGYAjN2PfX9) // [FAQ](https://links.morningbrew.com/c/t5?mbnlid=4VloMqduImvtGYAjN2PfX9)<br> Update your email preferences or unsubscribe [here](https://links.morningbrew.com/c/7sN?mbnlid=4VloMqduImvtGYAjN2PfX9). <br> View our privacy policy [here](https://links.morningbrew.com/c/t6?mbnlid=4VloMqduImvtGYAjN2PfX9). <br> Copyright \u00a9 2025 Morning Brew Inc. All rights reserved. <br> 22 W 19th St, 4th Floor, New York, NY 10011 | |\n\nBecome smarter in just 5 minutes\n\nMorning Brew delivers quick and insightful updates about the business world every day of the week from Wall St. to Silicon Valley.\n\nSubscribe\n\nBy subscribing, you accept our [Terms](https://www.morningbrew.com/terms-service) & [Privacy Policy](https://privacy.morningbrewinc.com/policies).\n\n![A mobile phone scrolling a newsletter issue of Morning Brew](<Base64-Image-Removed>)"
  },
  {
    "_id": "lCWqF141cRqxatHk3Xo9zI",
    "date": "2025-08-28",
    "slug": "dress-like-a-player",
    "content": "[Skip to main content](https://www.morningbrew.com/issues/dress-like-a-player#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### \u2615Dress like a player\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Nvidia gave a status update on the AI industry... |\n\n[![](https://media.sailthru.com/5z8/1k3/3/k/5c9296dc6e8c2.png)](http://www.morningbrew.com/)\n\n| |     |     |\n| --- | --- |\n| |     |     |\n| --- | --- |\n| August 28, 2025 | [View Online](https://links.morningbrew.com/c/DSR?mbnlid=lCWqF141cRqxatHk3Xo9zI) \\| [Sign Up](https://links.morningbrew.com/c/BJ_?mbnlid=lCWqF141cRqxatHk3Xo9zI) \\| [Shop](https://links.morningbrew.com/c/r-?mbnlid=lCWqF141cRqxatHk3Xo9zI) |\n\n|     |\n| --- |\n| [![Morning Brew](https://cdn.sanity.io/images/bl383u0v/production/65e2f7af234829dba0f11606a9a1430efab65892-1350x330.png?w=900&fit=max)](https://links.morningbrew.com/c/BK0?mbnlid=lCWqF141cRqxatHk3Xo9zI) |\n\n|     |     |\n| --- | --- |\n| Presented By | [![Pendulum](https://cdn.sanity.io/images/bl383u0v/production/60de72e1dc901d451021c37afc83b5b1a35e3634-2310x405.png?w=556&q=100&auto=format)](https://links.morningbrew.com/c/DQL?lp=logo&mbadid=99f94247f32b2d9c9902cfbabe2b4ec1&mbadv=a&mbnlid=lCWqF141cRqxatHk3Xo9zI) |\n\n|     |\n| --- |\n| **Duck!** The 20,000 tourists who descended on the small Spanish town of Bu\u00f1ol (usual population: ~10,000) saw red yesterday. That\u2019s because they were there for the 80th anniversary of the town\u2019s famous tomato tossing festival, known as the [Tomatina](https://links.morningbrew.com/c/DSS?mblid=6f7134e777b8&mbnlid=lCWqF141cRqxatHk3Xo9zI), where they chucked 120 tons of the squishy fruit at one another. Lest you worry that the tomatoes would have been better used to make a giant bowl of gazpacho rather than as fodder for the world\u2019s biggest food fight, organizers said the tomatoes thrown don\u2019t meet the standards for human consumption and would have been garbage anyway.<br>_\u2014Matty Merritt, Molly Liebergall, Dave Lozo, Abby Rubenstein_ | |\n\n| |     |\n| --- |\n| ### MARKETS |\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n|  |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/6097ec1c26b17401a61bfbfe4440c75ebfea1886-44x39.png) | Nasdaq | 21,590.14 | |     |\n| --- |\n| +0.21% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/6097ec1c26b17401a61bfbfe4440c75ebfea1886-44x39.png) | S&P | 6,481.40 | |     |\n| --- |\n| +0.24% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/6097ec1c26b17401a61bfbfe4440c75ebfea1886-44x39.png) | Dow | 45,565.23 | |     |\n| --- |\n| +0.32% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/e4a85d043f857cc8adaca9d546c099f6d7db37ba-44x39.png) | 10-Year | 4.238% | |     |\n| --- |\n| -2.0 bps | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/e4a85d043f857cc8adaca9d546c099f6d7db37ba-44x39.png) | Bitcoin | $111,369.07 | |     |\n| --- |\n| -0.28% | | |\n| |     |     |     |     |\n| --- | --- | --- | --- |\n| ![](https://cdn.sanity.io/images/bl383u0v/production/6097ec1c26b17401a61bfbfe4440c75ebfea1886-44x39.png) | Cracker Barrel | $62.33 | |     |\n| --- |\n| +8.01% | | |\n| |     |     |\n| --- | --- |\n| Data is provided by | [![](https://cdn.sanity.io/images/bl383u0v/production/61a0dac0b53d6766459afdca784ae8fe604f17ed-720x102.png?h=32&q=100&auto=format)](https://links.morningbrew.com/c/k2s?mbnlid=lCWqF141cRqxatHk3Xo9zI) |\n\n\\*Stock data as of market close, cryptocurrency data as of 7:00pm ET. [Here's what these numbers mean.](https://links.morningbrew.com/c/BK2?mbnlid=lCWqF141cRqxatHk3Xo9zI) |\n| - **Markets:**The S&P 500 clinched a [new record](https://links.morningbrew.com/c/DST?mblid=a058cba0f9d2&mbnlid=lCWqF141cRqxatHk3Xo9zI) yesterday, but the market\u2019s main event came after the bell when Nvidia released its earnings (more on that below). Cracker Barrel rose after regaining its old-timey charm by ditching the updated logo it released last week, which had spurred an outcry. |\n|  | |\n| --- |\n\n|     |\n| --- |\n| ### AI\n\n|     |\n| --- |\n| # [Nvidia sales rise, but so do its red flags](https://links.morningbrew.com/c/DTk?mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n|\n|\n\n[![Jensen Huang, CEO of Nvidia, speaks to journalists.](https://cdn.sanity.io/images/bl383u0v/production/f7ed8ed9154fab61c31f1e415e99c8b30a70139f-1500x1000.jpg?w=668&q=70&auto=format)](https://links.morningbrew.com/c/DTk?mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n_Picture alliance/Getty Images_\n\n|     |\n| --- |\n| As the rest of the stock market continued its earnings party, it peeked back at Nvidia, like a 20-year-and-11-month-old kid holding a beer in view of their dad at the BBQ looking for approval. At first glance, Nvidia [beat](https://links.morningbrew.com/c/DTl?mblid=35c34dbc7795&mbnlid=lCWqF141cRqxatHk3Xo9zI) most of Wall Street\u2019s expectations in revenue and profits, but there were also red flags that made investors antsy about a possible AI spending slowdown and pushed the chipmaker\u2019s stock down ~3% in after-hours trading.<br>With everyone looking to Nvidia\u2014which supplies key parts for AI companies\u2014as a litmus test for the health of the entire AI industry, the company reported solid growth and rising sales, with net income up 59%. But following explosive growth that propelled the company to a $4 trillion value, Nvidia offered a tepid revenue forecast.<br>And investors looking for cracks in the world\u2019s most valuable company\u2019s facade also noticed:<br>**Data centers missed the mark.** Nvidia\u2019s data center division\u2019s sales rose 56% compared to the same period last year. But with $41.1 billion in sales, Nvidia _just_ missed analyst estimates of $41.3 billion. Still, its data center unit is, alone, larger than any other chipmaker in the game and accounts for roughly 88% of the company\u2019s total sales.<br>**H20 was dry.** Questions linger over Nvidia\u2019s ability to corner the Chinese market, and it didn\u2019t sell its any of the lower-powered H20 chips it designed for China there last quarter, following the Trump administration\u2019s tightening of export restrictions. One exec said during Nvidia\u2019s earnings call that between $2 billion and $5 billion of H20 chips could be shipped to China if geopolitical tensions settle down. But semiconductors have become a key bargaining chip in the trade war with China.<br>**Big picture:** AI spending is a [major force](https://links.morningbrew.com/c/D1q?mblid=85c803313c90&mbnlid=lCWqF141cRqxatHk3Xo9zI) in the economy, with companies expected to spend $375 billion globally on AI infrastructure (like data centers) this year, according to UBS. That investment is expected to reach $500 billion by 2026. But some analysts worry that Nvidia\u2019s tapering earnings don\u2019t just represent the company\u2019s skyrocketing growth normalizing, but instead are the first sign that AI spending could be slowing. _\u2014MM_ |\n| |     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/75fb8a70db5f5bc6b7210cdde05dfa91eeb3f0ef-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DTm?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/f61048cc7cca6e313215d18ddc546bbb9f5279dd-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DTn?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/e4d40d61a934b028861fc21c8ae1f827b99638c5-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DTo?mbnlid=lCWqF141cRqxatHk3Xo9zI) | | | |\n\n|     |\n| --- |\n| ### WORLD\n\n|     |\n| --- |\n| # [Tour de headlines](https://www.morningbrew.com/issues/dress-like-a-player\\#)\n\n|\n|\n\n![The Annunciation Catholic Church is seen behind police tape following a mass shooting on August 27, 2025 in Minneapolis, Minnesota.](https://cdn.sanity.io/images/bl383u0v/production/d2b456ad0dd926e57f87a5e8455e7f905d094e93-1500x1000.jpg?w=668&q=70&auto=format)\n\n_Stephen Maturen/Getty Images_\n\n|     |\n| --- |\n| ![](https://em-content.zobj.net/thumbs/120/apple/237/broken-heart_1f494.png)**Two children killed in Minneapolis school shooting.** Another 17 people were injured, 14 of them children, and the shooter was also dead after he shot through the windows during Mass at Annunciation Catholic School, authorities [said](https://links.morningbrew.com/c/DSU?mblid=8625a3602d44&mbnlid=lCWqF141cRqxatHk3Xo9zI) yesterday. Two of the injured children were in critical condition but are expected to survive. The academic year for the school, which includes preschool through eighth grade, began on Monday. Authorities said they did not yet know the motive of the shooter, reportedly a former student of the school who was transgender, but FBI director Kash Patel said the shooting was being investigated as domestic terrorism and as a hate crime against Catholics.<br>**![](https://em-content.zobj.net/thumbs/120/apple/237/flag-for-india_1f1ee-1f1f3.png) Trump\u2019s 50% tariff on India kicked in.** India\u2019s exports to the US will now get hit with a steep [import tax](https://links.morningbrew.com/c/DSV?mblid=9551a790be86&mbnlid=lCWqF141cRqxatHk3Xo9zI), after President Trump doubled the 25% tariff he initially imposed on Indian goods to punish the country for buying Russian oil amid the war in Ukraine. The heightened tariff, which took effect yesterday, is putting a strain on US-India relations and could be a big blow to India since the US is its biggest export destination. It shipped about $87.3 billion worth of goods to the US last year. However, some major exports are exempt for now, including smartphones\u2014which is good news for Apple as it\u2019s been shifting manufacturing there in an effort to decrease its reliance on China.<br>**\u2695\ufe0f The FDA approved updated Covid vaccines\u2014but only for some people.** The regulator [signed off](https://links.morningbrew.com/c/DSW?mblid=cc9018952be6&mbnlid=lCWqF141cRqxatHk3Xo9zI) on this year\u2019s version of the shots; however, it revoked the broad authorization for the vaccines, placing new limits on who is eligible to get them. The approval is for those at higher risk of severe illness, including people over age 65 or those who have an underlying health condition. All others will now need to get approval from a doctor to get the jab. The change comes after Health and Human Services Secretary Robert F. Kennedy Jr. often criticized vaccines, especially those that use mRNA. It could impact who can access the shots and whether insurance companies will cover them. _\u2014AR_ | | |\n\n|     |\n| --- |\n| ### Presented By Pendulum\n\n# [**Au naturel GLP-1 support**](https://links.morningbrew.com/c/DQL?lp=title&mbadid=99f94247f32b2d9c9902cfbabe2b4ec1&mbadv=a&mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n[![Pendulum](https://cdn.sanity.io/images/bl383u0v/production/317dce62753a316b0324e2a6a7f08b0082ec2ba6-1000x750.jpg?w=668&q=90&auto=format)](https://links.morningbrew.com/c/DQL?lp=image&mbadid=99f94247f32b2d9c9902cfbabe2b4ec1&mbadv=a&mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n|     |\n| --- |\n| Did you hear about the group of PhD scientists from Johns Hopkins, Harvard, and Stanford who walked into a lab\u2014and left with a breakthrough probiotic in hand?<br>These scientists zeroed in on one of the most beneficial strains for gut health, _Akkermansia muciniphila_. They figured out how to bring live _Akkermansia muciniphila_ to the probiotics market. The final result of their work? Pendulum\u2019s [Akkermansia probiotic](https://links.morningbrew.com/c/DQL?lp=text1&mbadid=99f94247f32b2d9c9902cfbabe2b4ec1&mbadv=a&mblid=f954081e2c02&mbnlid=lCWqF141cRqxatHk3Xo9zI).<br>This particular strain is so groundbreaking because it can strengthen gut lining, [improve metabolic and digestive health](https://links.morningbrew.com/c/DQL?lp=text2&mbadid=99f94247f32b2d9c9902cfbabe2b4ec1&mbadv=a&mblid=5ed00a43132e&mbnlid=lCWqF141cRqxatHk3Xo9zI), and naturally increase GLP-1.\\*<br>With more than 16k medical provider recommendations, 4.5k mentions of _Akkermansia muciniphila_ in scientific publications, and loyal customers like Halle Berry, this stuff sure sounds promising.<br>Try Akkermansia yourself and [take 20% off](https://links.morningbrew.com/c/DQL?lp=text3&mbadid=99f94247f32b2d9c9902cfbabe2b4ec1&mbadv=a&mblid=cdcb65da8c3a&mbnlid=lCWqF141cRqxatHk3Xo9zI). | |\n\n|     |\n| --- |\n| ### TECH\n\n|     |\n| --- |\n| # [OpenAI says it\u2019ll do better amid suicide lawsuit](https://links.morningbrew.com/c/DSJ?mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n|\n|\n\n[![ChatGPT logo](https://cdn.sanity.io/images/bl383u0v/production/b206a3bb18a0bcdd8f1864070104c712f6e277fc-1500x1000.jpg?w=668&q=70&auto=format)](https://links.morningbrew.com/c/DSJ?mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n_Anna Kim_\n\n|     |\n| --- |\n| The maker of ChatGPT said in a blog post on Tuesday that it\u2019s \u201cworking to improve\u201d its safeguards, which can become \u201cless reliable in long interactions,\u201d after a wrongful death lawsuit was [filed](https://links.morningbrew.com/c/DOw?mblid=ce77a48d5e5e&mbnlid=lCWqF141cRqxatHk3Xo9zI) against OpenAI and CEO Sam Altman earlier that day.<br>California residents Matt and Maria Raine, whose son died by suicide in April, sued the people behind ChatGPT after discovering months\u2019 worth of conversations in which the chatbot appears to have encouraged their 16-year-old\u2019s suicidal ideation and dissuaded him from seeking outside help, the New York Times reported.<br>In the [blog post](https://links.morningbrew.com/c/DSK?mblid=73a74ed83b6d&mbnlid=lCWqF141cRqxatHk3Xo9zI), which didn\u2019t explicitly mention the lawsuit, OpenAI said:<br>- ChatGPT is supposed to respond to messages about self-harm with real-world resources. But \u201cas the back-and-forth grows\u2026it might eventually offer an answer that goes against our safeguards,\u201d which the company is \u201cworking to prevent.\u201d<br>- It\u2019s planning to add parental controls \u201csoon\u201d for ChatGPT accounts belonging to minors.<br>**Zoom out:** While the AI industry has been keen on chatbots as companions and therapists, there is mounting evidence of possible psychological danger. In two similar cases over the past year, a [14-year-old](https://links.morningbrew.com/c/t9y?mblid=a1c541308a3c&mbnlid=lCWqF141cRqxatHk3Xo9zI) and a 29-year-old also confided in other chatbots before taking their lives. This week, the American Psychiatric Association said ChatGPT, Google\u2019s Gemini, and Anthropic\u2019s Claude all [need](https://links.morningbrew.com/c/DSL?mblid=74eddd6e0d41&mbnlid=lCWqF141cRqxatHk3Xo9zI) \u201cfurther refinement\u201d in their responses to messages about self-harm. _\u2014ML_<br>_If you or someone you know needs help, call or text 988 to reach the Suicide and Crisis Lifeline, or visit [988lifeline.org](https://links.morningbrew.com/c/DSM?mblid=0f69f6f802b1&mbnlid=lCWqF141cRqxatHk3Xo9zI). Click [here](https://links.morningbrew.com/c/DSN?mblid=04dd14e60755&mbnlid=lCWqF141cRqxatHk3Xo9zI) for resources outside of the US._ |\n| |     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/75fb8a70db5f5bc6b7210cdde05dfa91eeb3f0ef-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DSO?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/f61048cc7cca6e313215d18ddc546bbb9f5279dd-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DSP?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/e4d40d61a934b028861fc21c8ae1f827b99638c5-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DSQ?mbnlid=lCWqF141cRqxatHk3Xo9zI) | | | |\n\n|     |\n| --- |\n| ### Together With West Capital Lending\n\n[![West Capital Lending](https://cdn.sanity.io/images/bl383u0v/production/7d936d027f0acde65fa4bee68b292868dbbeac10-2250x2250.heif?w=668&q=90&auto=format)](https://links.morningbrew.com/c/DQM?lp=image&mbadid=e42e70a53fe835c0b5a211adc24ddd90&mbadv=a&mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n|     |\n| --- |\n| **The HELOC whisperers.** They speak fluent cash-out and they don\u2019t just _process_ HELOCs; they understand them deeply. We\u2019re talking about West Capital Lending. They make [getting a HELOC as smooth as butter](https://links.morningbrew.com/c/DQM?lp=text1&mbadid=e42e70a53fe835c0b5a211adc24ddd90&mbadv=a&mblid=3055ae0d8c88&mbnlid=lCWqF141cRqxatHk3Xo9zI). Enjoy \u201cone-day closes\u201d with no drawn-out processes and fast, smart access to your own money. [Check eligibility](https://links.morningbrew.com/c/DQM?lp=text2&mbadid=e42e70a53fe835c0b5a211adc24ddd90&mbadv=a&mblid=b83db74c231e&mbnlid=lCWqF141cRqxatHk3Xo9zI). | |\n\n|     |\n| --- |\n| ### RETAIL\n\n|     |\n| --- |\n| # [American Eagle wants you to dress like Travis Kelce](https://links.morningbrew.com/c/DSX?mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n|\n|\n\n[![Travis Kelce and six others model clothing](https://cdn.sanity.io/images/bl383u0v/production/3be6e3e03f28e76e0a995b4bda9c671928869774-1500x1000.jpg?w=668&q=70&auto=format)](https://links.morningbrew.com/c/DSX?mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n_American Eagle_\n\n|     |\n| --- |\n| For a second day in a row, Travis Kelce announced a life-changing partnership. The future husband of Taylor Swift is the latest celebrity to [collaborate](https://links.morningbrew.com/c/DSY?mblid=8f858c0f6981&mbnlid=lCWqF141cRqxatHk3Xo9zI) with American Eagle on a clothing campaign as the denim wars rage on.<br>Unlike the \u201cSydney Sweeney Has Great Jeans\u201d ad that AE ~~had to pretend wasn\u2019t a genes pun~~ denied had anything to do with genes, the brand probably won\u2019t experience eugenics-related [blowback](https://links.morningbrew.com/c/DSZ?mblid=10ea3f8c2b5c&mbnlid=lCWqF141cRqxatHk3Xo9zI) from linking up with Tru Kolors, the clothing line of America\u2019s Gym Teacher. AE\u2019s stock jumped ~8.5% yesterday, although it\u2019s still down ~20% for the year.<br>Denim isn\u2019t the focus with AE x Tru Kolors, though. It\u2019s more about sporty-wear, like cricket sweaters, beanies, and utility cargo pants, aka dressing like Travis Kelce. But it\u2019s still a focus for AE\u2026and its competitors. After the Sweeney controversy, Gap and Lucky Brand dropped their own jeans-related ad blitzes that got attention both for their teamups with Katseye and Addison Rae, respectively, and their lack of anything that could be construed as commentary on genetics.<br>**Bottom line:** AE is trying to attract the Back-To-School Dollar, but its marketing efforts with Sweeney may have been [too far outside the box](https://links.morningbrew.com/c/DS-?mblid=f8ecb3564f12&mbnlid=lCWqF141cRqxatHk3Xo9zI) to help, according to Marketing Dive. With back-to-school spending expected to remain flat year over year, parents are focusing more on value and familiarity, and it\u2019s hard to find anyone more familiar to millennial parents and younger generations than Tayvis. _\u2014DL_ |\n| |     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/75fb8a70db5f5bc6b7210cdde05dfa91eeb3f0ef-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DS_?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/f61048cc7cca6e313215d18ddc546bbb9f5279dd-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DT0?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/e4d40d61a934b028861fc21c8ae1f827b99638c5-96x96.png?w=50&fit=max)](https://links.morningbrew.com/c/DT1?mbnlid=lCWqF141cRqxatHk3Xo9zI) | | | |\n\n|     |\n| --- |\n| ### STAT\n\n|     |\n| --- |\n| # [Prime number: These demon hunters are popular](https://www.morningbrew.com/issues/dress-like-a-player\\#)\n\n|\n|\n\n![Huntrix from KPop Demon Hunters](https://cdn.sanity.io/images/bl383u0v/production/870afd362cf9238a7eb20d7daab2535dcf5dc108-1200x675.jpg?w=668&q=70&auto=format)\n\n_Netflix_\n\n|     |\n| --- |\n| The girls of Huntrix are so talented that they not only slay monsters, they also slay the charts. _KPop Demon Hunters_ is now the [most-viewed](https://links.morningbrew.com/c/DT2?mblid=383f1cfc512b&mbnlid=lCWqF141cRqxatHk3Xo9zI) movie on Netflix ever, having racked up 236 million views since its June release. That pushed the Ryan Reynolds and Dwayne Johnson vehicle _Red Notice_ out of the No. 1 slot it had held for more than three years in Netflix\u2019s all-time rankings for English-language films. And that\u2019s not the only place the unstoppable animated girl group is [dominating](https://links.morningbrew.com/c/CrW?mblid=a25bed7ec2dc&mbnlid=lCWqF141cRqxatHk3Xo9zI):<br>- The sing-along version of the film released in theaters won the box office over the weekend, with The Hollywood Reporter estimating that it took in as much as $18 million.<br>- And the film\u2019s songs are certified hits. Billboard [said](https://links.morningbrew.com/c/DT3?mblid=c547b0c45b5f&mbnlid=lCWqF141cRqxatHk3Xo9zI) the soundtrack is the first one ever to generate four simultaneous top 10 hits on its Hot 100 list in the chart\u2019s 67-year history.<br>Netflix is keen to keep things golden: The streaming giant and the film\u2019s producer, Sony Pictures, are [in talks](https://links.morningbrew.com/c/DT4?mblid=2a5921d2efaa&mbnlid=lCWqF141cRqxatHk3Xo9zI) about making a sequel, per THR. _\u2014AR_ | | |\n\n|     |\n| --- |\n| ### Together With T-Mobile\n\n[![T-Mobile](https://cdn.sanity.io/images/bl383u0v/production/6f7a90f2335df572cfafcdfacf8a710ad916f399-1000x750.png?w=668&q=90&auto=format)](https://links.morningbrew.com/c/DQI?lp=image&mbadid=d71192b6a0a267c7c4e0e86717b25935&mbadv=a&mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n|     |\n| --- |\n| **Written in the stars.** Look up: Do ya see it? Probably not. But the mere _thought_ of T-Mobile\u2019s [T-Satellite](https://links.morningbrew.com/c/DQI?lp=text1&mbadid=d71192b6a0a267c7c4e0e86717b25935&mbadv=a&mblid=62236c9ae1e4&mbnlid=lCWqF141cRqxatHk3Xo9zI) working on your behalf up in space should be a comforting one. T-Mobile members with Experience Beyond plans can pay just [$10/line during this promo](https://links.morningbrew.com/c/DQI?lp=text2&mbadid=d71192b6a0a267c7c4e0e86717b25935&mbadv=a&mblid=d22cb02c0cfa&mbnlid=lCWqF141cRqxatHk3Xo9zI). Get in on the fastest median download speeds in the US. | |\n\n|     |\n| --- |\n| ### NEWS\n\n|     |\n| --- |\n| # [What else is brewing](https://www.morningbrew.com/issues/dress-like-a-player\\#)\n\n|\n|\n\n|     |\n| --- |\n| - [Centers for Disease Control and Prevention Director Susan Monarez](https://links.morningbrew.com/c/DT5?mblid=287168ab509b&mbnlid=lCWqF141cRqxatHk3Xo9zI)was fired yesterday after less than a month on the job, reportedly because she clashed with HHS Secretary RFK Jr. The firing came after Monarez refused to resign.<br>- [Denmark](https://links.morningbrew.com/c/DT7?mblid=6ad74f6be697&mbnlid=lCWqF141cRqxatHk3Xo9zI) summoned the top US diplomat there for talks, accusing the US of trying to drum up support in Greenland for the semiautonomous territory to secede from Denmark. President Trump has previously said he believes the US should control Greenland.<br>- [Microsoft employees](https://links.morningbrew.com/c/DTp?mblid=4e7a329995cc&mbnlid=lCWqF141cRqxatHk3Xo9zI) have been protesting the company\u2019s contracts and partnerships with the Israeli military and government, prompting a crackdown by their employer.<br>- [Kilmar \u00c1brego Garcia](https://links.morningbrew.com/c/DT8?mblid=0ab5d2d47b03&mbnlid=lCWqF141cRqxatHk3Xo9zI), a Salvadoran national whose mistaken deportation to El Salvador became a flashpoint in debates over President Trump\u2019s immigration crackdown, is seeking asylum in the US, hoping to avoid deportation to Uganda.<br>- [The Trump administration](https://links.morningbrew.com/c/DT9?mblid=0c5a32d1cbcf&mbnlid=lCWqF141cRqxatHk3Xo9zI) is taking over the management of Washington, DC\u2019s Union Station as the federal government seeks to exert power over the capital. Separately, [federal prosecutors](https://links.morningbrew.com/c/DTa?mblid=242ebb349086&mbnlid=lCWqF141cRqxatHk3Xo9zI) failed to secure an indictment against a man who threw a sandwich at federal officers in protest of their presence in the city. | | |\n\n|     |\n| --- |\n| ### RECS\n\n|     |\n| --- |\n| ![To-do list banner](https://cdn.sanity.io/images/bl383u0v/production/762cbc4c673b8613a0de1bf6711a4fc25276f7a4-1350x330.png?w=668&q=70&auto=format)\n\n|     |\n| --- |\n| ![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**Read:** You\u2019ve tried the McChicken and the Baja Blast, now [check out the history](https://links.morningbrew.com/c/DTb?mblid=7fe0e267736c&mbnlid=lCWqF141cRqxatHk3Xo9zI).\\*\\*<br>![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**Settle the debate:** Experts weigh in on whether you should leave your AC [on or off](https://links.morningbrew.com/c/DTc?mblid=98a539132ec2&mbnlid=lCWqF141cRqxatHk3Xo9zI).<br>![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**Get influenced:** Rolling Stone lists the [top creators](https://links.morningbrew.com/c/DTd?mblid=e3b14f220c7a&mbnlid=lCWqF141cRqxatHk3Xo9zI) of 2025.<br>![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png)**Forget the avocado toast:** [A map](https://links.morningbrew.com/c/DTe?mblid=58b7033f36bc&mbnlid=lCWqF141cRqxatHk3Xo9zI) of the world\u2019s most unaffordable housing markets.<br>**![](https://em-content.zobj.net/thumbs/120/apple/237/white-heavy-check-mark_2705.png) Deep research AI:** General LLMs give you summaries. [DeepSky\u2019s AI superagent](https://links.morningbrew.com/c/DQJ?mblid=a0c6f4a94146&mbnlid=lCWqF141cRqxatHk3Xo9zI) can give you strategic insight. Experience $20k+ in premium data sources. [Sign up for a free 30-day trial](https://links.morningbrew.com/c/DQJ?mblid=fd66fdf40881&mbnlid=lCWqF141cRqxatHk3Xo9zI) and get 200 reports. No CC required.\\*<br>_\\*A message from our sponsor. \\*\\*This is a product_ _recommendation from our writers. When you buy through this link, Morning Brew may earn a commission._ | | |\n\n|     |\n| --- |\n| ### PLAY\n\n|     |\n| --- |\n| [![Games available from Morning Brew](https://cdn.sanity.io/images/bl383u0v/production/3524949da8d220051120175f97246671d52e0b38-2700x660.png?w=668&q=70&auto=format)](https://links.morningbrew.com/c/AGP?mbnlid=lCWqF141cRqxatHk3Xo9zI)\n\n|     |\n| --- |\n| **Brew Mini:** Today\u2019s Mini is called \u201cTurntable,\u201d but don\u2019t let that confuse you\u2014just think of it as a Brew gamesception. [Play it here](https://links.morningbrew.com/c/DTf?mblid=0609e50bcfcc&mbnlid=lCWqF141cRqxatHk3Xo9zI), if you can wrap your mind around it.<br>## Three Headlines and a Lie<br>Three of these headlines are real and one is faker than 75% of the businesses run by the Real Housewives. Can you spot the odd one out?<br>1. Eric Adams advisor Winnie Greco [handed](https://links.morningbrew.com/c/DTg?mblid=72a6dfe5f15d&mbnlid=lCWqF141cRqxatHk3Xo9zI) a CITY reporter cash stuffed in a bag of potato chips<br>2. The call of a [native frog](https://links.morningbrew.com/c/DTh?mblid=ab24460f9a92&mbnlid=lCWqF141cRqxatHk3Xo9zI) is heard again in Southern California thanks to help from Mexico and AI<br>3. An 83-year-old pizza tycoon [fights](https://links.morningbrew.com/c/DTi?mblid=6484a710d45f&mbnlid=lCWqF141cRqxatHk3Xo9zI) to save 3,500 Domino\u2019s<br>4. The brides willing to [travel](https://links.morningbrew.com/c/DTj?mblid=247b9be9083a&mbnlid=lCWqF141cRqxatHk3Xo9zI) to Italy for controversial fish bones | | |\n\n|     |\n| --- |\n| ### SHARE THE BREW\n\n|     |\n| --- |\n| |     |\n| --- |\n| [Share Morning Brew](https://links.morningbrew.com/c/sH?mbnlid=lCWqF141cRqxatHk3Xo9zI) with your friends, acquire free Brew swag, and then acquire more friends as a result of your fresh Brew swag.<br>We\u2019re saying we\u2019ll give you free stuff and more friends if you share a link. One link.<br> [![](https://cdn.sanity.io/images/bl383u0v/production/e8c0d668fd2ce584105a14629f9409b29ddffc91-3840x2160.png?w=640&q=60)](https://links.morningbrew.com/c/sH?mbnlid=lCWqF141cRqxatHk3Xo9zI) <br>**Your referral count:** 5<br> [Click to Share](https://links.morningbrew.com/c/sH?mbnlid=lCWqF141cRqxatHk3Xo9zI) <br>Or copy & paste your referral link to others:<br>[morningbrew.com/daily/r/?kid=9ec4d467](https://links.morningbrew.com/c/7tc?mbnlid=lCWqF141cRqxatHk3Xo9zI) | | |\n\n|     |\n| --- |\n| ### ANSWER\n\n|     |\n| --- |\n| |     |\n| --- |\n| We made up the one about the brides.<br>Word of the Day<br>Today\u2019s Word of the Day is: blitzes, meaning \u201cfast and intensive campaigns or attacks.\u201d Thanks to Miranda W. from Philadelphia for the overwhelming suggestion. Submit another Word of the Day [here](https://links.morningbrew.com/c/aVn?mblid=55351d1e8c54&mbnlid=lCWqF141cRqxatHk3Xo9zI). | | |\n\n| |     |\n| --- |\n| **\u2722 A Note From Pendulum**<br>\\*Based on preclinical studies. This product is not intended for weight loss. | |\n| --- |\n\n|     |\n| --- |\n| [![](https://cdn.sanity.io/images/bl383u0v/production/365b8a21162a71a71c4190f5b1b6e2acd45a212f-96x96.png)](https://links.morningbrew.com/c/sJ?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/bc850f15719e478323219cbfe24640074ab0c306-96x96.png)](https://links.morningbrew.com/c/sK?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/4be3fbb7e388adf000065f63e01e069db9fdd60a-96x96.png)](https://links.morningbrew.com/c/sL?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/6af774b2e94fc201bff9ad736da144c7f1f8468a-96x96.png)](https://links.morningbrew.com/c/sM?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/ccfbd021e2000f116305d0f75c4e95dbdb2996ba-96x96.png)](https://links.morningbrew.com/c/49z?mbnlid=lCWqF141cRqxatHk3Xo9zI)[![](https://cdn.sanity.io/images/bl383u0v/production/56835eaa4b1e8e7314950c4862d5dfccc9fc410c-96x96.png)](https://links.morningbrew.com/c/sO?mbnlid=lCWqF141cRqxatHk3Xo9zI) |\n\n|     |\n| --- |\n| Written by [Abigail Rubenstein](https://links.morningbrew.com/c/t2R?mbnlid=lCWqF141cRqxatHk3Xo9zI), [Matty Merritt](https://links.morningbrew.com/c/sKN?mbnlid=lCWqF141cRqxatHk3Xo9zI), [Molly Liebergall](https://links.morningbrew.com/c/sKP?mbnlid=lCWqF141cRqxatHk3Xo9zI), and [Dave Lozo](https://links.morningbrew.com/c/sI-?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br>Was this email forwarded to you? Sign up [here](https://links.morningbrew.com/c/sJ8?mbnlid=lCWqF141cRqxatHk3Xo9zI).<br>**Take The Brew to work**<br>- Marketers: [Marketing Brew](https://links.morningbrew.com/c/6Ok?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br>- Corporate: [CFO Brew](https://links.morningbrew.com/c/6Ol?mbnlid=lCWqF141cRqxatHk3Xo9zI)[HR Brew](https://links.morningbrew.com/c/6Om?mbnlid=lCWqF141cRqxatHk3Xo9zI)[Revenue Brew](https://links.morningbrew.com/c/yJ0?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br>- Tech: [IT Brew](https://links.morningbrew.com/c/6On?mbnlid=lCWqF141cRqxatHk3Xo9zI)[Tech Brew](https://links.morningbrew.com/c/COA?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br>- Retailers: [Retail Brew](https://links.morningbrew.com/c/6Op?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br>- Healthcare: [Healthcare Brew](https://links.morningbrew.com/c/6Oq?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br>**Get smarter in just 5 minutes**<br>- Money & Career: [Money With Katie](https://links.morningbrew.com/c/s20?mbnlid=lCWqF141cRqxatHk3Xo9zI)[Bossy](https://links.morningbrew.com/c/s21?mbnlid=lCWqF141cRqxatHk3Xo9zI)[Brew Markets](https://links.morningbrew.com/c/s22?mbnlid=lCWqF141cRqxatHk3Xo9zI)[The Playbook](https://links.morningbrew.com/c/s23?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br>**Interested in podcasts?**<br>- Check out ours [here](https://links.morningbrew.com/c/s24?mbnlid=lCWqF141cRqxatHk3Xo9zI) |\n| [ADVERTISE](https://links.morningbrew.com/c/BK5?mbnlid=lCWqF141cRqxatHk3Xo9zI) // [CAREERS](https://links.morningbrew.com/c/BK6?mbnlid=lCWqF141cRqxatHk3Xo9zI) // [SHOP](https://links.morningbrew.com/c/t3?mbnlid=lCWqF141cRqxatHk3Xo9zI) // [FAQ](https://links.morningbrew.com/c/t5?mbnlid=lCWqF141cRqxatHk3Xo9zI)<br> Update your email preferences or unsubscribe [here](https://links.morningbrew.com/c/7sN?mbnlid=lCWqF141cRqxatHk3Xo9zI). <br> View our privacy policy [here](https://links.morningbrew.com/c/t6?mbnlid=lCWqF141cRqxatHk3Xo9zI). <br> Copyright \u00a9 2025 Morning Brew Inc. All rights reserved. <br> 22 W 19th St, 4th Floor, New York, NY 10011 | |\n\nBecome smarter in just 5 minutes\n\nMorning Brew delivers quick and insightful updates about the business world every day of the week from Wall St. to Silicon Valley.\n\nSubscribe\n\nBy subscribing, you accept our [Terms](https://www.morningbrew.com/terms-service) & [Privacy Policy](https://privacy.morningbrewinc.com/policies).\n\n![A mobile phone scrolling a newsletter issue of Morning Brew](<Base64-Image-Removed>)"
  }
]

print(next(archive_edition for archive_edition in archive if archive_edition["date"] == iso_date))


{'_id': 'lCWqF141cRqxatHk3Xo9zI', 'date': '2025-08-28', 'slug': 'dress-like-a-player', 'content': "[Skip to main content](https://www.morningbrew.com/issues/dress-like-a-player#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Dress like a player\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Nvidia gave a status update on the AI industry... |\

## Individual functions

In [68]:
# Selection logic for whether there will be a quiz to find in the edition

def selection_logic(date):
    if date.weekday() not in valid_weekdays:
        return False
    elif date.weekday() == 3 and date >= first_thursday:
        if date >= first_worded_slug:
            return True
        else:
            return True
    elif date.weekday() == 5 and date < first_thursday:
        return True
    else:
        return False

In [155]:
selection_logic(date)

True

In [70]:
# Extract the slug

def extract_slug(edition_metadata, date):
    if date.weekday() not in valid_weekdays:
        return None
    elif date.weekday() == 3 and date >= first_thursday:
        if date >= first_worded_slug:
            return edition_metadata["slug"]
        else:
            return edition_metadata["slug"].replace("/", "%2F")
    elif date.weekday() == 5 and date < first_thursday:
        return edition_metadata["slug"].replace("/", "%2F")

In [156]:
slug = extract_slug(edition_metadata, date)
print(slug)

spacex-ipo-ready-for-blastoff


In [72]:
# Scrape edition

def scrape_edition(slug):
    client = Firecrawl(api_key=api_key)
    scraped_edition = client.scrape(
        f"https://www.morningbrew.com/issues/{slug}",
        formats=["markdown"]
    )

    return scraped_edition



In [157]:
scraped_edition = scrape_edition(slug)
scraped_edition.markdown

'[Skip to main content](https://www.morningbrew.com/issues/spacex-ipo-ready-for-blastoff#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕SpaceX IPO ready for blastoff\n\nTo:Brew Readers\n\n* * *\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| |     |\n| --- |\n| Elon Musk\'s space company made its IPO plans public... |\n\n[![](https://

In [152]:
scraped_edition

Document(markdown='[Skip to main content](https://www.morningbrew.com/issues/its-just-field#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕It’s just Field\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Inflation reached 4.2% in May… |\n\n[![](https://media.sailthru.com/5z8/1k3/3/k/5c9296dc6e8c2.png)](http://www.morningbrew.com/)\n\n|     |  

In [121]:
# Cut everything before the quiz and break into parts

def cut_before_and_split(scraped_edition):
    # I decided to use a regex instead of the below as it's more robust
    # if date >= first_thursday:
    #     quiz_section_start_index = scraped_edition.markdown.lower().find("three headlines and") # Removed "a lie" as there was an edition with a typo missing out "a"
    # if date < first_thursday:
    #     quiz_section_start_index = scraped_edition.markdown.lower().find("saturday headlines")
    split_quiz = scraped_edition.markdown.split("<br>")
    # q1_index = re.compile(r'1\. (“)?.*\[([\w\s\.])*\](\(.*\))').search(scraped_edition.markdown.lower()).start()
    # quiz_section = scraped_edition.markdown[q1_index:]
    # return quiz_section.split("<br>")
    return split_quiz


In [143]:
q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.])*\](\(.*\))')
matched = q_search.search(parts[72])
matched

<re.Match object; span=(0, 121), match='2. The hottest [tourist attraction](https://links>

In [159]:
parts = cut_before_and_split(scraped_edition)
print([(i, item) for i, item in enumerate(parts)])

[(0, "[Skip to main content](https://www.morningbrew.com/issues/spacex-ipo-ready-for-blastoff#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕SpaceX IPO ready for blastoff\n\nTo:Brew Readers\n\n* * *\n\n|     |     |     |     |\n| --- | --- | --- | --- |\n| |     |\n| --- |\n| Elon Musk's space company made its IPO plans public... |\n\n[![](http

In [162]:
# Filter parts

def filter_parts(scraped_edition):
    split_quiz = scraped_edition.markdown.split("<br>")
    # Define what is being looked for to identify a question or answer and store only those parts
    # seeking = ["1. ", "2. ", "3. ", "4. "]
    q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.\-])*\](\(.*\))')
    answer_search = re.compile(r'(\banswer)(s?)(\b)')
    # answers_search = re.compile(r'\banswers\b')
    # hash_answer_search = re.compile(r'(\s### answer)(s)?') I tried this but some editions don't have "###"

    questions_and_answers = [item for item in split_quiz if q_search.search(item.lower()) or (answer_search.search(item.lower()) and item.startswith("[morningbrew.com"))]
    # return [item for item in parts if any(term in item.lower() for term in seeking) or hash_answer_search.search(item.lower())]

    if len(questions_and_answers) > 5:
        del questions_and_answers[5:]

    return questions_and_answers
    

# questions_and_answers = filter_parts(parts)
# questions_and_answers

In [163]:
questions_and_answers = filter_parts(scraped_edition)
questions_and_answers

['🅰️ **Harvard votes to limit A grades.** To crack down on grade inflation, the number of undergraduate A grades that Harvard University instructors can dole out will be [limited](https://www.theguardian.com/education/2026/may/20/harvard-college-cap-a-grade-students?utm_campaign=mb&utm_source=morning_brew&utm_medium=newsletter) to 20% of students in each course, plus an allowance for up to four additional students, beginning in fall 2027. While getting straight A’s will be tougher, there’ll be no limit on A-minus grades. Nearly 70% of Harvard faculty voted to approve the measure, according to results announced yesterday, though it’s overwhelmingly unpopular among students. For context, about two-thirds of letter grades given to Harvard undergraduates during the 2024–25 school year were A’s, up from roughly one-third in 2012–13. The [new cap](https://www.morningbrew.com/stories/harvard-deciding-whether-to-give-fewer-as?utm_campaign=mb&utm_source=morning_brew&utm_medium=newsletter) could

In [135]:
def clean_q4_and_answer(questions_and_answers):

    # Find the answer and extract and store it
    if len(questions_and_answers) == 4:
        answer_rough = questions_and_answers[3].lower().split("answer")[1]
        answer = answer_rough.strip(" |\n")
        questions_and_answers.append(answer)
        
    else:
        answer_index = questions_and_answers[4].lower().find("answer")
        answer_rough = questions_and_answers[4][answer_index:]
        answer_parts = answer_rough.split("|")
        answer_parts_stripped = [item.strip().strip("\n-") for item in answer_parts]
        answer_parts_stripped[0] = ""
        answer_list = []
        [answer_list.append(item) for item in answer_parts_stripped if item]
        answer = answer_list[0]
        questions_and_answers[4] = answer

    # Clean up question 4
    questions_and_answers[3] = (questions_and_answers[3].split("|"))[0]
    return questions_and_answers

# questions_and_answers = clean_q4_and_answer(questions_and_answers)
# questions_and_answers

In [136]:
questions_and_answers = clean_q4_and_answer(questions_and_answers)
questions_and_answers

['1. Tyga’s _$TARFACE_ album [receives](https://links.morningbrew.com/c/U6n?mblid=f92dc3fd9d90&mbnlid=slV5a3vuYndT7PDVh24Aj8) Pitchfork’s first 0/10 score in 19 years',
 '2. The hottest [tourist attraction](https://links.morningbrew.com/c/U6o?mblid=6222a3e47f01&mbnlid=slV5a3vuYndT7PDVh24Aj8) in Tonopah, NV, is this 74-year-old woman’s living room',
 '3. High school football player in Arkansas [practices](https://links.morningbrew.com/c/U6p?mblid=5ac37e49abf9&mbnlid=slV5a3vuYndT7PDVh24Aj8) with a venomous snake in his helmet',
 '4. Beth, the beloved vacuum of a 92-year-old Iowa senator, has [died](https://links.morningbrew.com/c/U6q?mblid=0d6a557bf158&mbnlid=slV5a3vuYndT7PDVh24Aj8) ',
 'we made up the one about the popular living room.']

In [137]:
# Headline text and url extraction

# def morning_brew_link_unpack(url):
#     response = requests.get(url)
#     return soup.find('link', attrs={'rel': 'canonical'}).get('href')
    

def url_headline_extract(index, questions_and_answers):
    url_re = re.compile(r'(\(.*\))')
    url = url_re.search(questions_and_answers[index]).group().strip("()")
    headline = questions_and_answers[index].replace(url, "").replace("()", "").replace("[", "").replace("]", "").lstrip("1234. “").rstrip("”")
    # print(headline)

    if "morningbrew" in url:
        url = urlsplit(requests.get(url, allow_redirects=False, timeout=10).headers.get("Location"))._replace(query="").geturl()

    # if any(term in headline for term in ["“", "”"]):
    #     headline_re = re.compile(r'(“.*”)')
    #     headline = headline_re.search(headline).group().strip("“”")
    # else:

    return headline, url

In [ ]:
quiz = {
    "id": id,
    "edition_date": date.isoformat(),
    "source_url": f"https://www.morningbrew.com/issues/{slug}",
    "headlines": [],
    "answer": questions_and_answers[4]
}

for index, item in enumerate(questions_and_answers):
    if item[0].isalpha():
        continue
    else:
        headline, url = url_headline_extract(index, questions_and_answers)
        quiz["headlines"].append({"text": headline, "url": url})

quiz

{'id': 'slV5a3vuYndT7PDVh24Aj8',
 'edition_date': '2026-08-13',
 'source_url': 'https://www.morningbrew.com/issues/what-about-bob',
 'headlines': [{'text': 'Tyga’s _$TARFACE_ album receives Pitchfork’s first 0/10 score in 19 years',
   'url': 'https://www.hotnewhiphop.com/1005560-tyga-starface-pitchfork-review'},
  {'text': 'The hottest tourist attraction in Tonopah, NV, is this 74-year-old woman’s living room',
   'url': 'https://giphy.com/gifs/CBSAllAccess-season-2-episode-7-sabrina-the-teenage-witch-kBH4I91xndN5pTPsIj'},
  {'text': 'High school football player in Arkansas practices with a venomous snake in his helmet',
   'url': 'https://apnews.com/article/venomous-snake-helmet-b4340a60e61757f937eb370b11b80098'},
  {'text': 'Beth, the beloved vacuum of a 92-year-old Iowa senator, has died ',
   'url': 'https://www.nytimes.com/2026/08/08/us/politics/beth-vacuum-chuck-grassley-iowa.html'}],
 'answer': 'we made up the one about the popular living room.'}

In [69]:
print(quiz)

{'id': 'slV5a3vuYndT7PDVh24Aj8', 'issue_date': datetime.date(2026, 8, 13), 'source_url': 'https://www.morningbrew.com/issues/what-about-bob', 'headlines': [{'text': 'Tyga’s _$TARFACE_ album receives Pitchfork’s first 0/10 score in 19 years', 'url': 'https://www.hotnewhiphop.com/1005560-tyga-starface-pitchfork-review'}, {'text': 'The hottest tourist attraction in Tonopah, NV, is this 74-year-old woman’s living room', 'url': 'https://giphy.com/gifs/CBSAllAccess-season-2-episode-7-sabrina-the-teenage-witch-kBH4I91xndN5pTPsIj'}, {'text': 'High school football player in Arkansas practices with a venomous snake in his helmet', 'url': 'https://apnews.com/article/venomous-snake-helmet-b4340a60e61757f937eb370b11b80098'}, {'text': 'Beth, the beloved vacuum of a 92-year-old Iowa senator, has died ', 'url': 'https://www.nytimes.com/2026/08/08/us/politics/beth-vacuum-chuck-grassley-iowa.html'}], 'answer': 'we made up the one about the popular living room.'}
